# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [3]:
import os
import tensorflow as tf
import numpy as np
import random

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

2025-05-11 23:02:46.488675: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-11 23:02:46.538193: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-11 23:02:46.921685: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-11 23:02:47.344083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746986567.747995   62263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746986567.83

## 1.0 Load and preprocess the data

In [ ]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [ ]:
def get_random_image(output_file="random_image.txt"):
    # Randomly select an image and its label
    index = random.randint(0, len(images) - 1)
    image = images[index].squeeze()  # (28, 28)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension (1, 28, 28)
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension (1, 28, 28, 1)
    
    label = np.argmax(labels[index])  # Get label
    
    # Save the image to a .txt file with 3 decimal places
    with open(output_file, "w") as f:
        for row in image.numpy().squeeze():  # Convert tensor to numpy and remove extra dimensions
            row_str = ".float " + ", ".join(f"{val:.3f}" for val in row)  # Using commas to separate values
            f.write(row_str + "\n")
    
    return image, label

### 2.2 Function to Print a tensor

Prints the shape and values of a tensor in a readable format.

For 4D tensors (e.g., batches of images), it prints the values of the first sample,
channel by channel. For 2D tensors (e.g., dense layer outputs), it prints all values
row by row. Other shapes are not currently supported.

In [ ]:
def print_shape_and_values(x):
    print(f"Shape: {x.shape}")
    
    # If it's a 4D tensor (e.g., batch of images), handle it
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                for j in range(width):
                    print(f"{x[0, i, j, c]:.3f}", end=" ")
                print()
            print()
    
    # If it's a 2D array (after flattening), handle it
    elif len(x.shape) == 2:
        rows, cols = x.shape
        for i in range(rows):
            for j in range(cols):
                print(f"{x[i, j]:.3f}", end=", ")
            print()
    else:
        print("Unsupported shape")


## 3.0 Load the model from mnist_cnn_model.keras

In [ ]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

## 4.0 Get a random image, label and step through the model layer by layer

### 4.1 Get a random image and label

In [ ]:
image, label = get_random_image()
print(f"Label: {label}")

Label: 9


### 4.2 Step through the model layer by layer

#### 4.2.1 Input Image

In [ ]:
print("Original Image:")
print_shape_and_values(image)

Original Image:
Shape: (1, 28, 28, 1)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.0

#### 4.2.2 Conv2D Layer

Note: Refer to section **4.2.1** for the input

In [16]:
conv2d_out = model.layers[0](image)
print_shape_and_values(conv2d_out)

Shape: (1, 24, 24, 8)
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.378 -0.229 -0.110 -0.067 -0.119 -0.250 -0.481 -0.560 -0.542 -0.504 -0.467 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.444 -0.317 -0.092 -0.015 -0.014 0.042 0.200 0.331 0.326 0.048 -0.274 -0.429 -0.510 -0.467 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.343 -0.086 -0.022 -0.121 -0.317 -0.375 -0.165 0.094 0.474 0.446 0.157 -0.142 -0.377 -0.480 -0.460 -0

#### 4.2.3 ReLU Activation

Note: Refer to section **4.2.2** for the input

In [17]:
relu_out = model.layers[1](conv2d_out)
print_shape_and_values(relu_out)

Shape: (1, 24, 24, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.042 0.200 0.331 0.326 0.048 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.094 0.474 0.446 0.157 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 

#### 4.2.4 MaxPooling

Note: Refer to section **4.2.3** for the input

In [18]:
maxpool_out = model.layers[2](relu_out)
print_shape_and_values(maxpool_out)

Shape: (1, 12, 12, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 

0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.200 0.474 0.446 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.285 0.000 0.000 0.153 0.154 0.000 0.000 0.000 
0.000 0.000 0.000 0.109 0.296 0.000 0.000 0.322 0.098 0.000 0.000 0.000 
0.000 0.000 0.000 0.119 0.405 0.000 0.141 0.385 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.044 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.184 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.127 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.083 0.000 0.000 0.000 

0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 
0.178 0.178 0.178 0.291 0.821 1.482 1.609 1.168 0.477 0.178 0.178 0.178 
0.178 0.178 0.351 1.322 2.266 3.262 3.383 2.438 1.510 0.423 0.178 0.

#### 4.2.5 Flatten

Note: Refer to section **4.2.4** for the input

In [19]:
flatten_out = model.layers[3](maxpool_out)
print_shape_and_values(flatten_out)

Shape: (1, 1152)
0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.291 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.821 0.000 0.000 0.163 0.070 0.323 0.000 0.000 1.482 0.000 0.000 0.364 0.000 0.508 0.000 0.000 1.609 0.000 0.000 0.081 0.000 0.107 0.000 0.000 1.168 0.000 0.101 0.000 0.000 0.000 0.000 0.000 0.477 0.000 0.126

#### 4.2.6 Fifth Layer: Dense Layer

Note: Refer to section **4.2.5** for the input

In [2]:
dense_out = model.layers[4](flatten_out)  # Fifth layer output
print_shape_and_values(dense_out)

NameError: name 'model' is not defined

#### 4.2.7 Layer Six: Softmax

Note: Refer to section **4.2.6** for the input

In [21]:
softmax_out = model.layers[5](dense_out)
print_shape_and_values(softmax_out)

Shape: (1, 10)
0.000 0.000 0.000 0.003 0.000 0.000 0.000 0.000 0.000 0.997 


### 4.3 Get model prediction

In [22]:
print(f"Predicted class: {np.argmax(softmax_out)}")
print(f"True class: {label}")

Predicted class: 9
True class: 9
